1️⃣ Download Sentinel-2 (10m bands)
2️⃣ Download IO LULC labels (same bbox, year)
3️⃣ Align + stack
4️⃣ Create 128×128 cubes
5️⃣ Train/Val/Test split
6️⃣ Fine-tune Clay using TerraTorch

In [ ]:
from pystac_client import Client
import planetary_computer
import stackstac
import numpy as np
from tqdm import tqdm
from dask.diagnostics import ProgressBar

In [22]:
#region - dehradun

bbox = [77.90, 30.20, 78.20, 30.45]


In [23]:

# Open Catalog
# ----------------------------------
catalog = Client.open("https://planetarycomputer.microsoft.com/api/stac/v1")

 

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2023-11-01/2023-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
)

items = list(search.get_items())
print("Sentinel scenes found:", len(items))

if len(items) == 0:
    raise ValueError("No Sentinel scenes found for given bbox/time/cloud filter.")


/opt/anaconda3/envs/odcenv/lib/python3.11/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Sentinel scenes found: 24


In [24]:
# Sign Items (with progress bar)
# ----------------------------------
signed_items = []
for item in tqdm(items, desc="Signing STAC items"):
    signed_items.append(planetary_computer.sign(item))




Signing STAC items: 100%|██████████| 24/24 [00:00<00:00, 30.92it/s]


In [25]:
#Stack Sentinel Bands
# ----------------------------------
print("Stacking bands...")

stack = stackstac.stack(
    signed_items,
    assets=["B02", "B03", "B04", "B08"],
    resolution=10,
    bounds_latlon=bbox,
    epsg=32644,
    chunksize=256,
)



print("Stack shape (lazy):", stack.shape)


Stacking bands...
Stack shape (lazy): (24, 4, 2848, 2959)


In [26]:

# ----------------------------------
# 5️⃣ Median Composite
# ----------------------------------
stack = stack.median(dim="time")

print("Computing median composite...")

from dask.diagnostics import ProgressBar
import time

for attempt in range(3):
    try:
        with ProgressBar():
            stack = stack.compute()
        break
    except Exception as e:
        print(f"Retry {attempt+1} due to error:", e)
        time.sleep(5)

# ----------------------------------
# 6️⃣ Final Array
# ----------------------------------
sentinel = stack.values.astype(np.float32)

print("Final Sentinel shape:", sentinel.shape)
print("Min/Max:", sentinel.min(), sentinel.max())

Computing median composite...
[#####                                   ] | 13% Completed | 35m 59ss
Retry 1 due to error: Error reading Window(col_off=512, row_off=0, width=256, height=256) from 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/44/R/KU/2023/11/15/S2A_MSIL2A_20231115T053101_N0509_R105_T44RKU_20231115T101556.SAFE/GRANULE/L2A_T44RKU_A043860_20231115T053059/IMG_DATA/R10m/T44RKU_20231115T053101_B08_10m.tif?st=2026-02-18T18%3A13%3A58Z&se=2026-02-19T18%3A58%3A58Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-02-19T04%3A28%3A13Z&ske=2026-02-26T04%3A28%3A13Z&sks=b&skv=2025-07-05&sig=oAXKbt5K6u4ie9TMNQtfyMEpgfEwl%2B9jJf0witK/B5E%3D': RasterioIOError('Read failed. See previous exception for details.')
[                                        ] | 0% Completed | 1.58 s ms
Retry 2 due to error: Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/43/R/GP/2023/11/20/S2B_MSIL2A_20231

RuntimeError: Error opening 'https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/44/R/KU/2023/12/05/S2A_MSIL2A_20231205T053211_N0509_R105_T44RKU_20231205T101113.SAFE/GRANULE/L2A_T44RKU_A044146_20231205T053206/IMG_DATA/R10m/T44RKU_20231205T053211_B04_10m.tif?st=2026-02-18T18%3A13%3A58Z&se=2026-02-19T18%3A58%3A58Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=2026-02-19T04%3A28%3A13Z&ske=2026-02-26T04%3A28%3A13Z&sks=b&skv=2025-07-05&sig=oAXKbt5K6u4ie9TMNQtfyMEpgfEwl%2B9jJf0witK/B5E%3D': RasterioIOError("'/vsicurl/https://sentinel2l2a01.blob.core.windows.net/sentinel2-l2/44/R/KU/2023/12/05/S2A_MSIL2A_20231205T053211_N0509_R105_T44RKU_20231205T101113.SAFE/GRANULE/L2A_T44RKU_A044146_20231205T053206/IMG_DATA/R10m/T44RKU_20231205T053211_B04_10m.tif?st=2026-02-18T18%3A13%3A58Z&se=2026-02-19T18%3A58%3A58Z&sp=rl&sv=2025-07-05&sr=c&skoid=9c8ff44a-6a2c-4dfb-b298-1c9212f64d9a&sktid=72f988bf-86f1-41af-91ab-2d7cd011db47&skt=20